# Day 13 — Gold Facts (corrected)

Builds fact tables from **corrected** Silver columns. Joins to Gold dims for surrogate keys.
Run `01_gold_dims_v1` first so dim tables exist.

| Fact | Source | Grain | Notes |
|---|---|---|---|
| FactChargingSession | api/sessions + api/payments | 1 row/session | cost reconciliation, session metrics |
| FactPayments | api/payments | 1 row/payment | gst_aud, refund_flag, payment method |
| FactComplaints | api/complaints | 1 row/complaint | resolution_days, sla_breach, priority |
| FactMaintenance | api/maintenance_events | 1 row/event | mttr_hours, root_cause |
| FactRealtimeSession | realtime/charging_sessions | 1 row/IoT session | device telemetry, power efficiency |

Silver columns actually available (post-fix):
- sessions: session_id, vehicle_id, station_id, customer_id, started_at, ended_at,
  duration_min, energy_kwh, cost_aud, peak_power_kw, connector_type, session_status,
  payment_id, created_at, updated_at
- payments: payment_id, session_id, customer_id, gateway, amount_aud, gst,
  payment_mode, status, processed_at, created_at, updated_at
- complaints: complaint_id, customer_id, station_id, category, priority,
  sla_breach, status, created_ts, resolved_ts, created_at, updated_at
- maintenance_events: event_id, charger_id, station_id, employee_id,
  root_cause, mttr_hours, event_ts, created_at, updated_at
- realtime/charging_sessions: session_id, charger_id, station_id, vehicle_id,
  customer_id, plug_in_ts, charge_end_ts, duration_min, energy_kwh, peak_kw,
  connector_type, session_status, tariff_id, raw_device_temp_c,
  signal_strength_dbm, firmware_ver, state_code, protocol_version


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
print('Imports OK')

In [ ]:
SILVER_API = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
SILVER_RT  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime'
GOLD_DIM   = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dims'
GOLD_FACT  = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/facts'
PIPELINE   = 'pl_gold_facts_v1'
RUN_TS     = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def silver(name):
    return spark.read.format('delta').load(f'{SILVER_API}/{name}')

def dim(name):
    return spark.read.format('delta').load(f'{GOLD_DIM}/{name}')

def write_fact(df, name, partition_cols=None):
    path = f'{GOLD_FACT}/{name}'
    w = (df.write.format('delta')
           .mode('overwrite')
           .option('overwriteSchema', 'true'))
    if partition_cols:
        w = w.partitionBy(*partition_cols)
    w.save(path)
    print(f'  {name:<30} {df.count():>7} rows -> {path}')

# time_key helper: YYYYMMDDHHH from any timestamp column
def time_key(ts):
    return F.concat_ws('',
        F.year(ts).cast('string'),
        F.lpad(F.month(ts).cast('string'),      2, '0'),
        F.lpad(F.dayofmonth(ts).cast('string'), 2, '0'),
        F.lpad(F.hour(ts).cast('string'),       2, '0'),
    )

print(f'GOLD FACT : {GOLD_FACT}\nRUN_TS    : {RUN_TS}')

In [ ]:
# Load dim surrogate key lookups (current rows only for SCD2 dims)
lk_station  = dim('DimStation').select('station_key',  'station_id')
lk_charger  = dim('DimCharger').filter(F.col('is_current') == True).select('charger_key',  'charger_id')
lk_customer = dim('DimCustomer').filter(F.col('is_current') == True).select('customer_key', 'customer_id')
lk_vehicle  = dim('DimVehicle').select('vehicle_key',  'vehicle_id')
lk_employee = dim('DimEmployee').select('employee_key', 'employee_id')
lk_tariff   = dim('DimTariff').filter(F.col('is_current') == True).select('tariff_key', 'tariff_id')
print('Dim lookups loaded')

In [ ]:
# FactChargingSession
# Grain: 1 row per session.
# Silver sessions cols used: session_id, vehicle_id, station_id, customer_id,
#   started_at, ended_at, duration_min, energy_kwh, cost_aud, peak_power_kw,
#   connector_type, session_status, payment_id
# Silver payments cols used: payment_id, amount_aud, gst, payment_mode, status (as payment_status)
# cost_reconciliation: MATCH if |cost_aud - amount_aud| < 0.01, MISMATCH otherwise, UNMATCHED if no payment

sessions_raw = silver('sessions')
payments_raw = silver('payments')

# One payment per session via payment_id on sessions (direct FK)
payment_lookup = payments_raw.select(
    F.col('payment_id'),
    F.col('amount_aud').alias('payment_amount_aud'),
    F.col('gst').alias('payment_gst'),
    F.trim(F.col('payment_mode')).alias('payment_mode'),
    F.trim(F.col('status')).alias('payment_status'),
    F.col('processed_at'),
)

fact_session = (
    sessions_raw
    .join(lk_station,  on='station_id',  how='left')
    .join(lk_customer, on='customer_id', how='left')
    .join(lk_vehicle,  on='vehicle_id',  how='left')
    .join(payment_lookup, on='payment_id', how='left')
    .withColumn('session_time_key', time_key(F.col('started_at')))
    .withColumn('session_date',  F.to_date('started_at'))
    .withColumn('session_year',  F.year('started_at'))
    .withColumn('session_month', F.month('started_at'))
    .withColumn('cost_reconciliation',
        F.when(F.col('payment_amount_aud').isNull(), F.lit('UNMATCHED'))
        .when(
            F.col('cost_aud').isNotNull() &
            (F.abs(F.col('payment_amount_aud') - F.col('cost_aud')) < 0.01),
            F.lit('MATCH')
        )
        .otherwise(F.lit('MISMATCH'))
    )
    .withColumn('cost_per_kwh',
        F.when(
            F.col('energy_kwh').isNotNull() & (F.col('energy_kwh') > 0) &
            F.col('cost_aud').isNotNull(),
            (F.col('cost_aud') / F.col('energy_kwh')).cast('decimal(8,4)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'session_id',
        'station_key', 'station_id',
        'customer_key', 'customer_id',
        'vehicle_key', 'vehicle_id',
        'payment_id',
        'session_time_key', 'session_date', 'session_year', 'session_month',
        F.col('started_at').alias('session_start_ts'),
        F.col('ended_at').alias('session_end_ts'),
        'duration_min', 'energy_kwh', 'cost_aud', 'peak_power_kw',
        F.trim(F.col('connector_type')).alias('connector_type'),
        F.trim(F.col('session_status')).alias('session_status'),
        'payment_amount_aud', 'payment_gst', 'payment_mode', 'payment_status',
        'processed_at', 'cost_per_kwh', 'cost_reconciliation',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_session, 'FactChargingSession', ['session_year', 'session_month'])
print('\nReconciliation distribution:')
fact_session.groupBy('cost_reconciliation').count().show()

In [ ]:
# FactPayments
# Grain: 1 row per payment.
# Silver cols used: payment_id, session_id, customer_id, gateway,
#   amount_aud, gst, payment_mode, status, processed_at
# refund_flag: status in (refunded, reversed)

fact_payments = (
    payments_raw
    .join(lk_customer, on='customer_id', how='left')
    .withColumn('payment_time_key', time_key(F.col('processed_at')))
    .withColumn('payment_year',  F.year('processed_at'))
    .withColumn('payment_month', F.month('processed_at'))
    .withColumn('refund_flag',
        F.when(
            F.lower(F.col('status')).isin('refunded', 'reversed'),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'payment_id', 'session_id',
        'customer_key', 'customer_id',
        'payment_time_key', 'payment_year', 'payment_month',
        F.col('processed_at').alias('payment_ts'),
        'amount_aud', 'gst',
        F.trim(F.col('gateway')).alias('gateway'),
        F.trim(F.col('payment_mode')).alias('payment_mode'),
        F.trim(F.col('status')).alias('payment_status'),
        'refund_flag',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_payments, 'FactPayments', ['payment_year', 'payment_month'])
print('\nPayment mode breakdown:')
fact_payments.groupBy('payment_mode', 'payment_status').count().orderBy('payment_mode').show()

In [ ]:
# FactComplaints
# Grain: 1 row per complaint.
# Silver cols used: complaint_id, customer_id, station_id, category,
#   priority, sla_breach, status, created_ts, resolved_ts, created_at, updated_at
# resolution_days: days from created_ts to resolved_ts (NULL if not resolved)

complaints_raw = silver('complaints')

fact_complaints = (
    complaints_raw
    .join(lk_customer, on='customer_id', how='left')
    .join(lk_station,  on='station_id',  how='left')
    .withColumn('complaint_time_key', time_key(F.col('created_ts')))
    .withColumn('complaint_year',  F.year('created_ts'))
    .withColumn('complaint_month', F.month('created_ts'))
    .withColumn('is_resolved',
        F.when(
            F.lower(F.col('status')).isin('resolved', 'closed'),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn('resolution_days',
        F.when(
            F.col('resolved_ts').isNotNull() & F.col('created_ts').isNotNull(),
            ((F.col('resolved_ts').cast('long') - F.col('created_ts').cast('long'))
             / 86400).cast('decimal(8,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'complaint_id',
        'customer_key', 'customer_id',
        'station_key',  'station_id',
        'complaint_time_key', 'complaint_year', 'complaint_month',
        F.col('created_ts').alias('complaint_created_ts'),
        F.col('resolved_ts').alias('complaint_resolved_ts'),
        F.trim(F.col('category')).alias('complaint_category'),
        F.trim(F.col('priority')).alias('priority'),
        F.col('sla_breach'),
        F.trim(F.col('status')).alias('complaint_status'),
        'is_resolved', 'resolution_days',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_complaints, 'FactComplaints', ['complaint_year', 'complaint_month'])
print('\nComplaints by priority & resolution:')
fact_complaints.groupBy('priority', 'is_resolved', 'sla_breach').count().orderBy('priority').show()

In [ ]:
# FactMaintenance
# Grain: 1 row per maintenance event.
# Silver cols used: event_id, charger_id, station_id, employee_id,
#   root_cause, mttr_hours, event_ts, created_at, updated_at

maintenance_raw = silver('maintenance_events')

fact_maintenance = (
    maintenance_raw
    .join(lk_charger,  on='charger_id',  how='left')
    .join(lk_station,  on='station_id',  how='left')
    .join(
        lk_employee.withColumnRenamed('employee_id', '_emp_id'),
        maintenance_raw['employee_id'] == F.col('_emp_id'),
        how='left'
    ).drop('_emp_id')
    .withColumn('event_time_key', time_key(F.col('event_ts')))
    .withColumn('event_year',  F.year('event_ts'))
    .withColumn('event_month', F.month('event_ts'))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'event_id',
        'charger_key', 'charger_id',
        'station_key', 'station_id',
        'employee_key', 'employee_id',
        'event_time_key', 'event_year', 'event_month',
        F.col('event_ts').alias('event_ts'),
        F.trim(F.col('root_cause')).alias('root_cause'),
        F.col('mttr_hours'),
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_maintenance, 'FactMaintenance', ['event_year', 'event_month'])
print('\nTop root causes:')
fact_maintenance.groupBy('root_cause').count().orderBy(F.col('count').desc()).show(10)

In [ ]:
# FactRealtimeSession
# Grain: 1 row per IoT/realtime charging session.
# Source: realtime/charging_sessions (blob Silver — fully correct, not affected by the cast bug).
# Silver cols: session_id, charger_id, station_id, vehicle_id, customer_id,
#   plug_in_ts, charge_end_ts, duration_min, energy_kwh, peak_kw,
#   connector_type, session_status, tariff_id, raw_device_temp_c,
#   signal_strength_dbm, firmware_ver, state_code, protocol_version

rt_sessions = spark.read.format('delta').load(f'{SILVER_RT}/charging_sessions')

fact_realtime = (
    rt_sessions
    .join(lk_charger,  on='charger_id',  how='left')
    .join(lk_station,  on='station_id',  how='left')
    .join(lk_customer, on='customer_id', how='left')
    .join(lk_vehicle,  on='vehicle_id',  how='left')
    .join(lk_tariff,   on='tariff_id',   how='left')
    .withColumn('session_time_key', time_key(F.col('plug_in_ts')))
    .withColumn('session_year',  F.year('plug_in_ts'))
    .withColumn('session_month', F.month('plug_in_ts'))
    # power_efficiency_pct = actual_kwh / (peak_kw × hours) × 100
    .withColumn('power_efficiency_pct',
        F.when(
            F.col('peak_kw').isNotNull() & (F.col('peak_kw') > 0) &
            F.col('duration_min').isNotNull() & (F.col('duration_min') > 0),
            (F.col('energy_kwh') / (F.col('peak_kw') * F.col('duration_min') / 60) * 100
            ).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'session_id',
        'charger_key', 'charger_id',
        'station_key', 'station_id',
        'customer_key', 'customer_id',
        'vehicle_key', 'vehicle_id',
        'tariff_key', 'tariff_id',
        'session_time_key', 'session_year', 'session_month',
        F.col('plug_in_ts').alias('session_start_ts'),
        F.col('charge_end_ts').alias('session_end_ts'),
        'duration_min', 'energy_kwh',
        F.col('peak_kw').alias('peak_power_kw'),
        F.trim(F.col('connector_type')).alias('connector_type'),
        F.trim(F.col('session_status')).alias('session_status'),
        F.col('raw_device_temp_c').alias('device_temp_c'),
        'signal_strength_dbm',
        F.col('firmware_ver').alias('firmware_version'),
        F.trim(F.col('state_code')).alias('state_code'),
        F.trim(F.col('protocol_version')).alias('protocol_version'),
        'power_efficiency_pct',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_realtime, 'FactRealtimeSession', ['session_year', 'session_month'])
print('\nAvg efficiency by firmware:')
fact_realtime.groupBy('firmware_version').agg(
    F.count('session_id').alias('sessions'),
    F.round(F.avg('power_efficiency_pct'), 2).alias('avg_efficiency_pct'),
    F.round(F.avg('device_temp_c'), 2).alias('avg_temp_c'),
).orderBy('firmware_version').show()

In [ ]:
# Summary
FACTS = [
    'FactChargingSession',
    'FactPayments',
    'FactComplaints',
    'FactMaintenance',
    'FactRealtimeSession',
]
print('=' * 60)
print('GOLD FACTS SUMMARY')
print('=' * 60)
for f in FACTS:
    path = f'{GOLD_FACT}/{f}'
    try:
        cnt = spark.read.format('delta').load(path).count()
        print(f'  {f:<30} {cnt:>8} rows')
    except Exception as e:
        print(f'  {f:<30} ERROR: {e}')
print(f'\nRUN_TS : {RUN_TS}')